# Tidy IPEDS salary data

This notebook reads `Data_10-5-2026.csv`, removes the UNC System institutions that are not regular universities, and reshapes faculty count and salary outlay columns into one tidy row per institution, year, and faculty group.

## Imports and paths

In [ ]:
from pathlib import Path
import re

import pandas as pd

process_dir = Path.cwd()
input_path = process_dir / "Data_10-5-2026.csv"
output_path = process_dir / "ipeds_salary_tidy.csv"

input_path, output_path

## Load and filter source data

In [ ]:
raw = pd.read_csv(input_path)

excluded_unit_ids = {
    199184,  # University of North Carolina School of the Arts
    433174,  # Carolinas College of Health Sciences
}

filtered = raw[~raw["UnitID"].isin(excluded_unit_ids)].copy()

raw.shape, filtered.shape

## Reshape faculty counts and salary outlays

In [ ]:
unit_col = "UnitID"
institution_col = "Institution Name"

count_pattern = re.compile(
    r"^Instructional staff on 9  10  11 or 12 month contract-total "
    r"\(SAL(\d{4})_IS(?:_RV)?  (.+)\)$"
)
salary_outlay_pattern = re.compile(
    r"^Salary outlays - total "
    r"\(SAL(\d{4})_IS(?:_RV)?  (.+)\)$"
)

rank_to_group = {
    "All full-time instructional staff total": "All",
    "Professor": "Prof",
    "Associate professor": "Assoc",
    "Assistant professor": "Assist",
    "Instructor": "Instructor",
    "Lecturer": "Lecturer",
}

def matching_columns(pattern):
    cols = []
    for col in filtered.columns:
        match = pattern.match(col)
        if match and match.group(2) in rank_to_group:
            cols.append(col)
    return cols

def reshape_measure(columns, pattern, value_name):
    long = filtered.melt(
        id_vars=[unit_col, institution_col],
        value_vars=columns,
        var_name="measure",
        value_name=value_name,
    )
    measure_parts = long["measure"].str.extract(pattern)
    long["year"] = measure_parts[0].astype(int)
    long["group"] = measure_parts[1].map(rank_to_group)
    long[value_name] = pd.to_numeric(long[value_name], errors="coerce")
    return long[[unit_col, institution_col, "year", "group", value_name]]

count_cols = matching_columns(count_pattern)
salary_outlay_cols = matching_columns(salary_outlay_pattern)

if not count_cols:
    raise ValueError("No IPEDS faculty count columns were found.")
if not salary_outlay_cols:
    raise ValueError("No IPEDS salary outlay columns were found.")

long_counts = reshape_measure(count_cols, count_pattern, "count")
long_outlays = reshape_measure(salary_outlay_cols, salary_outlay_pattern, "salary_outlay")

long_salary = long_counts.merge(
    long_outlays,
    on=[unit_col, institution_col, "year", "group"],
    how="outer",
    validate="one_to_one",
)

rank_salary = long_salary[long_salary["group"].isin(["Prof", "Assoc", "Assist", "All"])].copy()
nontt_salary = (
    long_salary[long_salary["group"].isin(["Instructor", "Lecturer"])]
    .groupby([unit_col, institution_col, "year"], as_index=False)[["count", "salary_outlay"]]
    .sum(min_count=1)
)
nontt_salary["group"] = "nontt"

tidy = pd.concat(
    [rank_salary[[unit_col, institution_col, "year", "group", "count", "salary_outlay"]], nontt_salary],
    ignore_index=True,
)
tidy["avg_salary"] = tidy["salary_outlay"] / tidy["count"]

tidy.head()

## Add institution abbreviations

In [ ]:
unit_id_to_abbreviation = {
    197869: "ASU",
    198464: "ECU",
    198507: "ECSU",
    198543: "FSU",
    199102: "NC A&T",
    199157: "NCCU",
    199193: "NCSU",
    199111: "UNCA",
    199120: "UNC",
    199139: "UNCC",
    199148: "UNCG",
    199281: "UNCP",
    199218: "UNCW",
    200004: "WCU",
    199999: "WSSU",
}

tidy["abbreviation"] = tidy[unit_col].map(unit_id_to_abbreviation)

missing_abbreviations = tidy.loc[tidy["abbreviation"].isna(), [unit_col, institution_col]].drop_duplicates()
if not missing_abbreviations.empty:
    raise ValueError(f"Missing abbreviations for: {missing_abbreviations.to_dict(orient='records')}")

tidy.head()

## Final ordering and save

In [ ]:
group_order = pd.CategoricalDtype(["Prof", "Assoc", "Assist", "nontt", "All"], ordered=True)

tidy = tidy.rename(columns={institution_col: "institution_name"})
tidy["group"] = tidy["group"].astype(group_order)
tidy["count"] = tidy["count"].astype("Int64")
tidy["salary_outlay"] = tidy["salary_outlay"].astype("Int64")

tidy = tidy[
    ["UnitID", "year", "group", "institution_name", "abbreviation", "count", "salary_outlay", "avg_salary"]
].sort_values(["UnitID", "year", "group"]).reset_index(drop=True)
tidy["group"] = tidy["group"].astype(str)

tidy.to_csv(output_path, index=False)

tidy.shape, output_path

## Quick checks

In [ ]:
assert not tidy["UnitID"].isin(excluded_unit_ids).any()
assert tidy["abbreviation"].notna().all()
assert set(tidy["group"]) == {"Prof", "Assoc", "Assist", "nontt", "All"}
assert tidy.loc[tidy["count"].notna() & tidy["salary_outlay"].notna(), "avg_salary"].notna().all()

tidy.groupby("group", sort=False).size()